<a href="https://colab.research.google.com/github/Rajeraghav/AI-Engineer-Journey/blob/main/Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
# ============================================================
#                 AI CHATBOT ASSISTANT
#          CSV DATASET + NLP + MACHINE LEARNING
# ============================================================

# Install required libraries if needed
!pip install -q pandas numpy scikit-learn nltk

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import os
import random

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK resources
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


# ============================================================
# 1. UPLOAD CSV FILE AT RUNTIME
# ============================================================

print("=" * 60)
print("             AI CHATBOT ASSISTANT")
print("=" * 60)

print("\nPlease upload your CSV file.")
print("Expected columns:")
print("intent, text, response")
print()

uploaded = files.upload()

# Get uploaded filename automatically
filename = list(uploaded.keys())[0]

print("\nUploaded file:", filename)


# ============================================================
# 2. LOAD CSV DATASET
# ============================================================

df = pd.read_csv(filename)

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = ["intent", "text", "response"]

# Remove unwanted spaces from column names
df.columns = df.columns.str.strip().str.lower()

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}\n"
        f"Your CSV must contain: {required_columns}"
    )

print("\nRequired columns found successfully.")


# ============================================================
# 4. KEEP ONLY REQUIRED COLUMNS
# ============================================================

df = df[["intent", "text", "response"]].copy()


# ============================================================
# 5. REMOVE MISSING VALUES
# ============================================================

print("\nMissing values before cleaning:")
print(df.isnull().sum())

df = df.dropna(subset=["intent", "text", "response"])

# Convert everything to string
df["intent"] = df["intent"].astype(str)
df["text"] = df["text"].astype(str)
df["response"] = df["response"].astype(str)

print("\nDataset after removing missing values:")
print(df.shape)


# ============================================================
# 6. REMOVE DUPLICATE QUESTIONS
# ============================================================

df = df.drop_duplicates(subset=["text"])

print("Dataset after removing duplicates:")
print(df.shape)


# ============================================================
# 7. TEXT PREPROCESSING
# ============================================================

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords and lemmatize
    processed_words = []

    for word in words:

        if word not in stop_words:

            word = lemmatizer.lemmatize(word)

            processed_words.append(word)

    return " ".join(processed_words)


# Apply preprocessing
df["clean_text"] = df["text"].apply(preprocess_text)


# ============================================================
# 8. DISPLAY DATASET
# ============================================================

print("\nSample processed data:")
print(df[["intent", "text", "clean_text", "response"]].head())


# ============================================================
# 9. CHECK NUMBER OF INTENTS
# ============================================================

number_of_intents = df["intent"].nunique()

print("\nNumber of unique intents:", number_of_intents)

print("\nAvailable intents:")
print(df["intent"].unique())


# ============================================================
# 10. CHECK INTENT DISTRIBUTION
# ============================================================

print("\nIntent distribution:")
print(df["intent"].value_counts())


# ============================================================
# 11. PREPARE X AND Y
# ============================================================

X = df["clean_text"]
y = df["intent"]


# ============================================================
# 12. TRAIN / TEST SPLIT
# ============================================================

# Check whether stratified split is possible
intent_counts = y.value_counts()

if len(df) >= 10 and intent_counts.min() >= 2:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

else:

    print(
        "\nDataset is small or some intents have only one example."
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )


# ============================================================
# 13. CREATE NLP + MACHINE LEARNING PIPELINE
# ============================================================

model = Pipeline([

    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=10000,
            sublinear_tf=True
        )
    ),

    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            C=5
        )
    )
])


# ============================================================
# 14. TRAIN MODEL
# ============================================================

print("\nTraining chatbot model...")

model.fit(X_train, y_train)

print("Model training completed successfully!")


# ============================================================
# 15. TEST MODEL
# ============================================================

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

print(f"Accuracy: {accuracy * 100:.2f}%")


# ============================================================
# 16. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

try:
    print(classification_report(y_test, y_pred, zero_division=0))
except:
    print("Classification report could not be generated.")

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
print(pd.DataFrame(cm, index=model.classes_, columns=model.classes_))

# ============================================================
# 17. CREATE INTENT -> RESPONSES DICTIONARY
# ============================================================

intent_responses = {}

for intent in df["intent"].unique():

    responses = df[
        df["intent"] == intent
    ]["response"].tolist()

    intent_responses[intent] = responses


# ============================================================
# 18. CHATBOT RESPONSE FUNCTION
# ============================================================

def chatbot_response(user_input):

    # Check empty input
    if not user_input or not user_input.strip():

        return "Please enter a question."


    # Preprocess user input
    clean_input = preprocess_text(user_input)


    # Predict intent probabilities
    probabilities = model.predict_proba(
        [clean_input]
    )[0]

    # Get highest probability
    max_probability = np.max(probabilities)

    # Get predicted intent
    predicted_intent = model.predict(
        [clean_input]
    )[0]


    # ========================================================
    # CONFIDENCE THRESHOLD
    # ========================================================

    # If model is not confident enough
    if max_probability < 0.25:

        return (
            "I'm sorry, I don't understand that yet. "
            "Could you please rephrase your question?"
        )


    # ========================================================
    # GET RESPONSE
    # ========================================================

    responses = intent_responses.get(
        predicted_intent,
        []
    )

    if not responses:

        return (
            "I'm sorry, I don't have a response "
            "for that question yet."
        )


    # Randomly select one response
    response = random.choice(responses)

    return response


# ============================================================
# 19. SHOW CHATBOT INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("CHATBOT READY")
print("=" * 60)

print("Number of training examples:", len(df))
print("Number of intents:", number_of_intents)

print("\nType 'exit' to stop the chatbot.")
print("=" * 60)


# ============================================================
# 20. START CHATBOT
# ============================================================

while True:

    user_input = input("\nYou: ")

    # Exit command
    if user_input.lower().strip() in [
        "exit",
        "quit",
        "bye"
    ]:

        print("Bot: Goodbye! Have a nice day!")
        break


    # Generate chatbot response
    response = chatbot_response(user_input)

    print("Bot:", response)

             AI CHATBOT ASSISTANT

Please upload your CSV file.
Expected columns:
intent, text, response



Saving ai_chatbot_dataset.csv to ai_chatbot_dataset (4).csv

Uploaded file: ai_chatbot_dataset (4).csv

Dataset loaded successfully!
Dataset shape: (203, 3)

Column names:
['intent', 'text', 'response']

Required columns found successfully.

Missing values before cleaning:
intent      0
text        0
response    0
dtype: int64

Dataset after removing missing values:
(203, 3)
Dataset after removing duplicates:
(203, 3)

Sample processed data:
     intent            text      clean_text  \
0  greeting           Hello           hello   
1  greeting              Hi              hi   
2  greeting             Hey             hey   
3  greeting    Good morning    good morning   
4  greeting  Good afternoon  good afternoon   

                              response  
0           Hello! How can I help you?  
1              Hi! How can I help you?  
2               Hey! Nice to meet you.  
3    Good morning! How can I help you?  
4  Good afternoon! How can I help you?  

Number of unique intents